# CNN-LSTM Smart Insole — Gait Estimation
## Leave-One-Subject-Out (LOSO) 3-Fold Cross-Validation

Each fold: one subject is held out for testing, the remaining two train the model from scratch.
Results are reported per fold and aggregated.

> **Note:** Subjects 1 and 3 were excluded due to battery-induced timestamp drift causing insole–frame desynchronisation. Evaluation uses LOSO on the three remaining subjects, following standard practice for small-cohort gait datasets.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Paths ─────────────────────────────────────────────────────────────────────
# Set BASE to the folder that contains the 'synced' and 'models_3' subdirectories.
BASE       = Path("data")          # <-- change to your data root
SYNCED_DIR = BASE / "synced"
MODELS_DIR = BASE / "models_3"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Synced dir:", SYNCED_DIR)
print("Models dir:", MODELS_DIR)


## 2. Joint setup and skeleton connections

In [ ]:
TARGET_JOINTS = [
    "left-hip",
    "right-hip",
    "left-knee",
    "right-knee",
    "left-heel",
    "right-heel",
    "left-big-toe-tip",
    "right-big-toe-tip",
]
OUTPUT_JOINTS = TARGET_JOINTS.copy()

CONNECTIONS = [
    ("left-hip", "left-knee"),
    ("left-knee", "left-heel"),
    ("left-heel", "left-big-toe-tip"),
    ("right-hip", "right-knee"),
    ("right-knee", "right-heel"),
    ("right-heel", "right-big-toe-tip"),
    ("left-hip", "right-hip"),
]

def as_str_list(names):
    return [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in names]

## 3. Load synced sessions

In [ ]:
def load_synced_sessions(synced_dir: Path):
    session_files = sorted(synced_dir.glob("*_synced.npz"))
    if not session_files:
        raise FileNotFoundError(f"No synced files found in {synced_dir}")
    sessions = []
    for path in session_files:
        data = np.load(path, allow_pickle=True)
        insoles     = np.asarray(data["insoles"],     dtype=np.float32)
        joints      = np.asarray(data["joints"],      dtype=np.float32)
        joint_names = as_str_list(data["joint_names"])
        timestamps  = np.asarray(data["timestamps"],  dtype=np.float64)
        sessions.append({
            "name":        path.stem.replace("_synced", ""),
            "insoles":     insoles,
            "joints":      joints,
            "joint_names": joint_names,
            "timestamps":  timestamps,
        })
        print(f"{path.name}: insoles={insoles.shape}, joints={joints.shape}")
    return sessions

sessions = load_synced_sessions(SYNCED_DIR)
print(f"\nTotal subjects loaded: {len(sessions)}")
assert len(sessions) >= 2, (
    f"Need at least 2 subjects for LOSO, found {len(sessions)}. "
    "Check SYNCED_DIR."
)

## 4. Session preprocessing

In [ ]:
def replace_nan_with_zero(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_joint_index(joint_names):
    mapping = {name: i for i, name in enumerate(joint_names)}
    missing = [name for name in TARGET_JOINTS if name not in mapping]
    if missing:
        raise ValueError(f"Missing required joints: {missing}")
    return mapping

def select_output_targets(joints, joint_names):
    name_to_idx = build_joint_index(joint_names)
    keep_indices = [name_to_idx[name] for name in OUTPUT_JOINTS]
    targets = joints[:, keep_indices, :]
    return targets.astype(np.float32)

def preprocess_sessions(sessions):
    processed = []
    for session in sessions:
        x = replace_nan_with_zero(session["insoles"])
        y = select_output_targets(session["joints"], session["joint_names"])
        x_norm = x.astype(np.float32)
        y_norm = y.astype(np.float32)
        processed.append({
            "name":       session["name"],
            "X":          x_norm.astype(np.float32),
            "Y":          y_norm,
            "Y_raw":      y.astype(np.float32),
            "timestamps": session["timestamps"],
        })
        print(f"{session['name']}: X={x_norm.shape}, Y={y_norm.shape}")
    return processed

processed_sessions = preprocess_sessions(sessions)

## 5. Model configuration

In [ ]:
SEQ_LEN          = 30
SEQ_STRIDE       = 1
BATCH_SIZE       = 32
NUM_WORKERS      = 2
CNN_CHANNELS     = (16, 32)
LSTM_HIDDEN_SIZE = 256
LSTM_LAYERS      = 2
DROPOUT          = 0.3
LR               = 1e-3
WEIGHT_DECAY     = 1e-5
EPOCHS           = 400
PATIENCE         = 20
VAL_RATIO        = 0.15   # fraction of train sequences reserved for validation / early stopping

print("Config OK")
print(f"  SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}")
print(f"  CNN_CHANNELS={CNN_CHANNELS}, LSTM_HIDDEN_SIZE={LSTM_HIDDEN_SIZE}, LSTM_LAYERS={LSTM_LAYERS}")

## 6. Build temporal sequences

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, sessions, items, seq_len):
        self.sessions = sessions
        self.items    = items
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        session_idx, end_idx = self.items[idx]
        s = self.sessions[session_idx]
        start_idx = end_idx - self.seq_len + 1
        # Extract temporal window (no velocity, only raw pressure)
        # Shape: (T, 2, H, W)
        x     = s["X"][start_idx : end_idx + 1]
        # Target at last timestep
        y     = s["Y"][end_idx]
        y_raw = s["Y_raw"][end_idx]
        return {
            "X":       torch.from_numpy(x).float(),
            "Y":       torch.from_numpy(y).float(),
            "Y_raw":   torch.from_numpy(y_raw).float(),
            "session": s["name"],
        }


def build_items(sessions_subset, seq_len):
    """Build (session_idx, end_idx) pairs for a subset of sessions."""
    items = []
    for local_i, s in enumerate(sessions_subset):
        T = len(s["X"])
        if T < seq_len:
            print(f"  WARNING: session {s['name']} has only {T} frames "
                  f"< SEQ_LEN={seq_len}, skipped.")
            continue
        for end_idx in range(seq_len - 1, T):
            items.append((local_i, end_idx))
    return items

print("Dataset helpers defined.")

## 7. CNN-LSTM model

In [ ]:
# ── Temporal attention ──────────────────────────────────────
class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        scores  = self.attn(lstm_out)               # (B, T, 1)
        weights = torch.softmax(scores, dim=1)       # (B, T, 1)
        context = (weights * lstm_out).sum(dim=1)    # (B, hidden_size)
        return context

# ── Frame encoder ───────────────────────────────────────────
class FrameEncoder(nn.Module):
    def __init__(self, in_channels=2, c1=16, c2=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(),
            nn.Conv2d(c1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2),
            nn.ReLU(),
            nn.Conv2d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),   # (batch, c2, 1, 1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, start_dim=1)  # (batch, c2) = (batch, 32)
        return x

# ── CNN-LSTM + Attention ─────────────────────────────────────
class CNNLSTMRegressor(nn.Module):
    def __init__(self, output_dim, c1=16, c2=32,
                 hidden_size=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = FrameEncoder(in_channels=2, c1=c1, c2=c2)
        with torch.no_grad():
            dummy = torch.zeros(1, 2, 66, 30)
            encoder_dim = self.encoder(dummy).shape[1]
            print("ENCODER DIM =", encoder_dim)   # 32

        self.proj      = nn.Linear(encoder_dim, 256)
        self.lstm      = nn.LSTM(256, hidden_size, lstm_layers,
                                 dropout=dropout if lstm_layers > 1 else 0.0,
                                 batch_first=True)
        self.attention = TemporalAttention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.encoder(x)
        x = self.proj(x)
        x = x.view(B, T, -1)
        x, _ = self.lstm(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# ── Helpers ──────────────────────────────────────────────────
def make_model():
    output_dim = len(OUTPUT_JOINTS) * 3
    m = CNNLSTMRegressor(
        output_dim  = output_dim,
        c1          = CNN_CHANNELS[0],
        c2          = CNN_CHANNELS[1],
        hidden_size = LSTM_HIDDEN_SIZE,
        lstm_layers = LSTM_LAYERS,
        dropout     = DROPOUT,
    ).to(DEVICE)
    return m

def make_optimizer(model):
    return torch.optim.Adam([
        {"params": model.encoder.parameters(),   "lr": 2e-3},
        {"params": model.lstm.parameters(),       "lr": 3e-3},
        {"params": model.fc.parameters(),         "lr": 1e-3},
        {"params": model.attention.parameters(),  "lr": 1e-3},
    ])

# Sanity check
_m = make_model()
print(_m)
del _m

## 8. Training and evaluation helpers

In [ ]:
criterion = nn.SmoothL1Loss()

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            x    = batch["X"].to(DEVICE)
            y    = batch["Y"].to(DEVICE).view(batch["X"].size(0), -1)
            pred = model(x)
            loss = criterion(pred, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            bs = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
    return total_loss / max(n_samples, 1)


def evaluate(model, loader, y_mean=None, y_std=None):
    """Return loss, MSE, MAE, per-joint MAE, and raw pred/true arrays."""
    model.eval()
    total_loss, n_samples = 0.0, 0
    preds_list, trues_list = [], []
    with torch.no_grad():
        for batch in loader:
            x     = batch["X"].to(DEVICE)
            y     = batch["Y"].to(DEVICE).view(x.size(0), -1)
            y_raw = batch["Y_raw"].cpu().numpy()
            pred  = model(x)
            loss  = criterion(pred, y)
            bs    = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
            pred_raw = pred.view(bs, len(OUTPUT_JOINTS), 3).cpu().numpy()
            if y_mean is not None and y_std is not None:
                pred_raw = pred_raw * y_std + y_mean
            preds_list.append(pred_raw)
            trues_list.append(y_raw)

    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    mse   = float(np.mean((preds - trues) ** 2))
    mae   = float(np.mean(np.abs(preds - trues)))
    per_joint_mae = np.mean(np.abs(preds - trues), axis=(0, 2))  # (J,)
    return {
        "loss":          total_loss / max(n_samples, 1),
        "mse":           mse,
        "mae":           mae,
        "per_joint_mae": per_joint_mae,
        "preds":         preds,
        "trues":         trues,
    }

print("Helpers defined.")

## 9. LOSO 3-Fold Cross-Validation

**Protocol:**
- Fold *k*: subject *k* → test set (never seen during training)
- Remaining 2 subjects → training pool
- `VAL_RATIO` fraction of training sequences withheld for validation / early stopping
- Model weights re-initialised from scratch at each fold


In [ ]:
n_subjects     = len(processed_sessions)
fold_results   = []   # one dict per fold
fold_histories = []   # loss curves for plotting

print("=" * 60)
print("LOSO Cross-Validation")
print("=" * 60)

for test_subj_idx in range(n_subjects):

    test_name   = processed_sessions[test_subj_idx]["name"]
    train_idxs  = [i for i in range(n_subjects) if i != test_subj_idx]
    train_names = [processed_sessions[i]["name"] for i in train_idxs]

    print(f"\n{'─' * 60}")
    print(f"Fold {test_subj_idx + 1}/{n_subjects}  |  "
          f"Test: [{test_name}]  |  Train: {train_names}")
    print(f"{'─' * 60}")

    # ── Session subsets ──────────────────────────────────────
    train_sessions = [processed_sessions[i] for i in train_idxs]
    test_sessions  = [processed_sessions[test_subj_idx]]

    train_sessions = [copy.deepcopy(s) for s in train_sessions]
    test_sessions  = [copy.deepcopy(s) for s in test_sessions]

    train_Y = np.concatenate([s["Y"] for s in train_sessions], axis=0)
    Y_mean  = train_Y.mean(axis=0)
    Y_std   = train_Y.std(axis=0) + 1e-6

    for s in train_sessions:
        s["Y"] = (s["Y"] - Y_mean) / Y_std
    for s in test_sessions:
        s["Y"] = (s["Y"] - Y_mean) / Y_std

    # ── Sliding-window item lists ────────────────────────────
    all_train_items = build_items(train_sessions, SEQ_LEN)
    test_items      = build_items(test_sessions,  SEQ_LEN)

    # Shuffle then carve out validation fraction
    random.seed(SEED)
    random.shuffle(all_train_items)
    n_val            = max(1, int(len(all_train_items) * VAL_RATIO))
    val_items_fold   = all_train_items[-n_val:]
    train_items_fold = all_train_items[:-n_val]

    print(f"  Train sequences : {len(train_items_fold)}")
    print(f"  Val   sequences : {len(val_items_fold)}")
    print(f"  Test  sequences : {len(test_items)}")

    # ── DataLoaders ──────────────────────────────────────────
    kw = dict(batch_size=BATCH_SIZE, num_workers=0,
              pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(
        SlidingWindowDataset(train_sessions, train_items_fold, SEQ_LEN),
        shuffle=True, **kw)
    val_loader = DataLoader(
        SlidingWindowDataset(train_sessions, val_items_fold, SEQ_LEN),
        shuffle=False, **kw)
    test_loader = DataLoader(
        SlidingWindowDataset(test_sessions, test_items, SEQ_LEN),
        shuffle=False, **kw)

    # ── Fresh model and optimiser ────────────────────────────
    torch.manual_seed(SEED)
    model      = make_model()
    optimizer  = make_optimizer(model)
    model_path = MODELS_DIR / f"cnn_lstm_fold{test_subj_idx+1}_{test_name}.pt"

    # ── Training loop with early stopping ───────────────────
    history       = {"train_loss": [], "val_loss": []}
    best_val      = float("inf")
    best_epoch    = -1
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer)
        val_loss   = run_epoch(model, val_loader,   optimizer=None)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"  Epoch {epoch:02d}/{EPOCHS} | "
              f"train={train_loss:.6f} | val={val_loss:.6f}", end="")

        if val_loss < best_val:
            best_val      = val_loss
            best_epoch    = epoch
            patience_left = PATIENCE
            torch.save(model.state_dict(), model_path)
            print("  ✓ saved", end="")
        else:
            patience_left -= 1
        print()

        if patience_left <= 0:
            print(f"  Early stopping triggered.")
            break

    print(f"  Best epoch: {best_epoch}, best val loss: {best_val:.6f}")

    # ── Load best checkpoint and evaluate on test set ───────
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    results = evaluate(model, test_loader, y_mean=Y_mean, y_std=Y_std)

    print(f"\n  ── Fold {test_subj_idx+1} Test Results ──────────────")
    print(f"  Test subject  : {test_name}")
    print(f"  Test loss     : {results['loss']:.6f}")
    print(f"  Test MSE      : {results['mse']:.6f}")
    print(f"  Test MAE      : {results['mae']:.6f}")
    print(f"  Per-joint MAE :")
    for jname, jmae in zip(OUTPUT_JOINTS, results["per_joint_mae"]):
        print(f"    {jname:<22s}: {jmae:.4f}")

    fold_results.append({
        "fold":       test_subj_idx + 1,
        "test_name":  test_name,
        "best_epoch": best_epoch,
        "best_val":   best_val,
        **{k: results[k] for k in
           ("loss", "mse", "mae", "per_joint_mae", "preds", "trues")},
    })
    fold_histories.append(history)

## 10. Aggregated results across folds

In [ ]:
mses    = [r["mse"] for r in fold_results]
maes    = [r["mae"] for r in fold_results]
pj_maes = np.stack([r["per_joint_mae"] for r in fold_results])  # (3, J)

print("=" * 60)
print("LOSO Cross-Validation — Aggregated Results")
print("=" * 60)

print(f"\n{'Fold':<6} {'Subject':<16} {'MSE':>10} {'MAE':>10}")
print("─" * 46)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['test_name']:<16} "
          f"{r['mse']:>10.6f} {r['mae']:>10.6f}")
print("─" * 46)
print(f"  {'Mean':<20} {np.mean(mses):>10.6f} {np.mean(maes):>10.6f}")
print(f"  {'Std':<20} {np.std(mses):>10.6f} {np.std(maes):>10.6f}")

print("\nPer-joint MAE — mean ± std across folds:")
for j, jname in enumerate(OUTPUT_JOINTS):
    vals = pj_maes[:, j]
    print(f"  {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}")

## 14. LaTeX-ready summary table (copy into report)

In [ ]:
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcc}")
print(r"\toprule")
print(r"Test Subject & MSE & MAE \\")
print(r"\midrule")
for r in fold_results:
    print(f"  {r['test_name']} (Fold {r['fold']}) & "
          f"{r['mse']:.4f} & {r['mae']:.4f} \\\\")
print(r"\midrule")
print(f"  Mean $\\pm$ Std & "
      f"{np.mean(mses):.4f} $\\pm$ {np.std(mses):.4f} & "
      f"{np.mean(maes):.4f} $\\pm$ {np.std(maes):.4f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{LOSO 5-fold cross-validation results. "
      r"MSE and MAE are computed on normalised joint coordinates.}")
print(r"\label{tab:loso_results}")
print(r"\end{table}")